# List Comprehensions
**Topic:** Python Fundamentals

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


---
## What you'll explore

By the end of this demo you will be able to:

- **Translate** any `for` loop that builds a list into an equivalent list comprehension
- **Explain** how the optional filter condition `if` works inside a comprehension
- **Interpret** dictionary and set comprehensions as extensions of the same pattern

---
## How we got here

In *05: String Methods* we applied `.strip().lower()` chains to Series columns. Before pandas, the same work would have been done with a `for` loop that builds a list. In *02: Control Flow* we wrote those loops by hand. List comprehensions are Python's shorthand for exactly that pattern: loop + optional filter + transform, all in one readable line.

---
## Why this matters for data science

List comprehensions appear constantly in feature engineering: generating column names dynamically, selecting numeric columns by dtype, creating derived feature lists, and filtering rows before a DataFrame is even built. They are also the entry point to generator expressions, which power memory-efficient data loading for large datasets. Many pandas and scikit-learn idioms rely on comprehension-style column selection such as `[c for c in df.columns if c.startswith("feat_")]`.

---
## Try it yourself

In [2]:
# ▶ Run this cell and observe the output.
# Then try changing the values and running again.

# Traditional for loop
squares_loop = []
for x in range(10):
    squares_loop.append(x ** 2)

# Equivalent list comprehension
squares_comp = [x ** 2 for x in range(10)]

print('for loop:      ', squares_loop)
print('comprehension: ', squares_comp)
print('equal?', squares_loop == squares_comp)

for loop:       [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
comprehension:  [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
equal? True


In [ ]:
# ✏️ Your turn — modify this code:
# 1. Change the condition from > 5000 to > 10000 and see how many salaries remain
# 2. Add a transformation: instead of filtering, convert each salary to thousands
# 3. What happens if you remove the if condition entirely?

salaries = [32000, 75000, 48000, 92000, 61000, 5500, 110000]

high_earners = [s for s in salaries if s > 50000]
normalized   = [round(s / max(salaries), 3) for s in salaries]

print('high earners:', high_earners)
print('normalized:  ', normalized)

In [ ]:
# 🎯 Challenge:
# Rewrite this for loop as a single list comprehension:
#
# result = []
# for x in range(20):
#     if x % 3 == 0:
#         result.append(x ** 2)
#
# Hint: list comprehensions can include an if clause at the end

# Your code here:

---
## What's happening?

A list comprehension builds a new list by expressing the loop, the optional filter, and the transformation on a single line, in the order you read them left to right.

| Part | Syntax position | What it does |
|------|----------------|-------------|
| Expression | First | The value to include in the new list |
| `for` clause | Middle | Iterates over the input sequence |
| `if` clause | Last (optional) | Keeps only items where the condition is true |

```python
# For loop version
result = []
for score in scores:
    if score >= threshold:
        result.append(round(score, 2))

# List comprehension version (identical output)
result = [round(score, 2) for score in scores if score >= threshold]
```

### The same pattern extends to dicts and sets

```python
# Dict comprehension: column name to mean value
col_means = {col: df[col].mean() for col in df.select_dtypes("number").columns}

# Set comprehension: unique departments, uppercased
depts = {d.upper() for d in df["department"] if pd.notna(d)}
```

### When not to use a comprehension

If the body of the loop is more than one or two operations, or if it has side effects (writing to a file, printing), use a plain `for` loop. Comprehensions are for building values, not for executing actions.

Return to the widget and switch to "transform + filter" mode to see the most common comprehension shape in data science code.

---
## A direct example: capping outliers with one line

Eight quiz scores, three entered above 100 by mistake. One comprehension fixes all of them.

- **Notice:** The expression inside the comprehension (`min(s, 100)`) runs once per score — same idea as calling a function on every row
- **Notice:** The original list is unchanged; the comprehension produces a new list
- **Notice:** This is the same operation as pandas `.clip(upper=100)`, written in plain Python

In [ ]:
students      = ["S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8"]
raw_scores    = [112, 88, 55, 97, 103, 72, 68, 91]
clipped_scores = [min(s, 100) for s in raw_scores]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(students, raw_scores, color="#E45756")
axes[0].axhline(100, linestyle="--", color="black", linewidth=0.8, label="cap = 100")
axes[0].set_title("Raw scores (some above 100)", fontsize=13)
axes[0].set_ylabel("Score")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

axes[1].bar(students, clipped_scores, color="#4C72B0")
axes[1].axhline(100, linestyle="--", color="black", linewidth=0.8)
axes[1].set_title("After [min(s, 100) for s in raw_scores]", fontsize=13)
axes[1].set_ylabel("Score")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

---
## Real-world example: Selecting and renaming features for a model

Before training a model, data scientists often select a subset of columns, normalize their names, and filter out low-variance features. The chart shows this three-step pipeline applied to a dataset with 20 candidate features, using comprehensions at each step.

Notice:

- **Notice:** The selection step (`[c for c in cols if c.startswith("feat_")]`) produces a clean subset without any temporary variables
- **Notice:** The rename step (`{c: c.replace("feat_", "") for c in selected}`) is a dict comprehension that maps old names to new ones, passed directly into `df.rename()`
- **Notice:** The variance filter step removes features whose standard deviation falls below a threshold, a common pre-processing step before running PCA or regularized regression

> **Discussion question:** A teammate proposes a pandas-only version using `.filter()` and `.rename()` method chains instead of comprehensions. When would you prefer the comprehension approach, and when would the method-chain version be clearer?

In [ ]:
np.random.seed(88)

# ── Feature selection pipeline using list and dict comprehensions ──────────────
n_rows, n_feats = 200, 20
all_cols  = [f"feat_{i:02d}" for i in range(n_feats)] + ["target", "id_col", "raw_text"]
df_full   = pd.DataFrame(
    np.random.randn(n_rows, len(all_cols)), columns=all_cols
)
for col in ["feat_07", "feat_12", "feat_15"]:
    df_full[col] = df_full[col] * 0.01

feature_cols  = [c for c in df_full.columns if c.startswith("feat_")]
std_threshold = 0.1
selected      = [c for c in feature_cols if df_full[c].std() > std_threshold]
dropped       = [c for c in feature_cols if c not in selected]
rename_map    = {c: c.replace("feat_", "f") for c in selected}
stds          = df_full[feature_cols].std().sort_values()

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["#E45756" if c in dropped else "#4C72B0" for c in stds.index]
ax.bar(stds.index, stds.values, color=colors)
ax.axhline(std_threshold, linestyle="--", color="#F58518",
           label=f"Variance threshold = {std_threshold}")
ax.set_title(
    f"Feature Standard Deviations — {len(selected)} kept, {len(dropped)} dropped (low variance)",
    fontsize=12)
ax.set_xlabel("Feature")
ax.set_ylabel("Standard Deviation")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


### Comprehension patterns in data science code

| Task | Comprehension | Equivalent pandas |
|------|--------------|------------------|
| Select numeric columns | `[c for c in df if df[c].dtype != object]` | `df.select_dtypes("number").columns` |
| Build feature name list | `[f"lag_{i}" for i in range(1, 8)]` | Manual list or `pd.Series.add_prefix()` |
| Filter low-variance cols | `[c for c in cols if df[c].std() > thr]` | `df.loc[:, df.std() > thr]` |
| Map labels to integers | `{label: i for i, label in enumerate(classes)}` | `LabelEncoder().fit_transform()` |
| Flatten a list of lists | `[x for sublist in nested for x in sublist]` | `pd.Series(nested).explode()` |

---
## Key takeaway

> **A list comprehension is a for-loop and an optional filter collapsed into one readable line; it is the most common Python shorthand in feature engineering and column selection code.**

---
*Next up: Lambda, Map, Filter & Apply — anonymous functions and the functional-style tools that power pandas `.apply()` workflows*